# Parker Aerospace: Next-Quarter Component Failure Forecast

**Question the business owner asks** (VP Aftermarket & Customer Support, Parker Aerospace): *which installed Parker components will generate a HIGH/CRITICAL service event or warranty claim next quarter, so Field Service can intervene before the operator raises an AOG?*

**Why this notebook replaces the earlier part-failure model.** The earlier model scored AUC 0.999, but its label (`failure_risk_category`) is a threshold rule on `health_index`, which was also a feature. The model was re-learning the rule, not forecasting. This notebook fixes that:

| Design choice | Why |
|---|---|
| **Time-based split**: features use only service records before `2026-06-01`; label is any HIGH/CRITICAL anomaly or warranty-claim-risk event from `2026-06-01` to `2026-08-31` | No information from the label window can leak into features |
| **No derived health/risk columns** (`health_index`, `failure_risk_category`, `days_to_predicted_failure`, exposure columns) | They are rule-based summaries computed using the outcome period |
| **Stratified 5-fold CV** with balanced sample weights | Rare positive class (~3.5%), so a single 80/20 split is too noisy |
| **Business metric = top-decile capture** | Field Service can proactively inspect ~10% of the fleet per quarter; what matters is how many next-quarter events that list catches |

Output: `genie_zeroops_mfg_catalog.default.ml_component_failure_forecast` (one row per component).

In [0]:
import pandas as pd, numpy as np

C   = "genie_zeroops_mfg_catalog.default"
CUT = "2026-06-01"   # features strictly before, label on/after
OUT = f"{C}.ml_component_failure_forecast"

feat = spark.sql(f"""
WITH hist AS (SELECT * FROM {C}.bronze_maintenance_records WHERE maintenance_date <  '{CUT}'),
fut  AS (SELECT component_id,
                max(CASE WHEN (anomaly_flag AND anomaly_severity IN ('HIGH','CRITICAL')) OR warranty_claim_risk THEN 1 ELSE 0 END) AS y,
                count(CASE WHEN (anomaly_flag AND anomaly_severity IN ('HIGH','CRITICAL')) OR warranty_claim_risk THEN 1 END)        AS n_future_events
         FROM {C}.bronze_maintenance_records WHERE maintenance_date >= '{CUT}' GROUP BY 1),
h AS (SELECT component_id,
             count(*)                                                    AS n_services,
             avg(cast(is_overdue AS INT))                                AS overdue_rate,
             max(days_since_last_service)                                AS max_gap,
             avg(days_since_last_service)                                AS avg_gap,
             sum(cast(anomaly_flag AS INT))                              AS n_anom,
             sum(CASE WHEN anomaly_severity IN ('HIGH','CRITICAL') THEN 1 ELSE 0 END) AS n_anom_hc,
             sum(cast(warranty_claim_risk AS INT))                       AS n_warr,
             avg(CASE WHEN compliance_status = 'COMPLIANT' THEN 1 ELSE 0 END) AS compliance_rate,
             max(cycles_at_service)                                      AS cycles_last,
             datediff('{CUT}', max(maintenance_date))                    AS days_since_last,
             avg(CASE WHEN flight_delay_risk IN ('HIGH','CRITICAL') THEN 1 ELSE 0 END) AS delay_rate,
             count(DISTINCT mro_shop)                                    AS n_shops
      FROM hist GROUP BY 1)
SELECT r.component_id, r.component_type, r.criticality, r.unit_cost_usd, r.maintenance_interval_days, r.max_cycles,
       r.airline, r.tail_number, datediff('{CUT}', r.install_date) AS install_age_days,
       a.age_years AS ac_age, a.total_cycles AS ac_cycles,
       h.* EXCEPT (component_id),
       coalesce(f.y, 0) AS y, coalesce(f.n_future_events, 0) AS n_future_events
FROM {C}.bronze_components_registry r
JOIN {C}.bronze_parker_aircraft_registry a USING (tail_number)
LEFT JOIN h USING (component_id)
LEFT JOIN fut f USING (component_id)
""").toPandas()

feat["cycle_life_used"]               = feat.cycles_last / feat.max_cycles
feat["days_since_last_over_interval"] = feat.days_since_last / feat.maintenance_interval_days

ID_COLS  = ["component_id", "tail_number"]
CAT_COLS = ["component_type", "criticality", "airline"]
X = pd.get_dummies(feat.drop(columns=ID_COLS + ["y", "n_future_events"]), columns=CAT_COLS).fillna(0).astype(float)
y = feat["y"].values
print(f"{len(feat)} components | {X.shape[1]} features | next-quarter event rate {y.mean():.1%} ({y.sum()} components)")

In [0]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {
    "gbt":    GradientBoostingClassifier(random_state=42, max_depth=2, n_estimators=150, learning_rate=0.05, subsample=0.8),
    "logreg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced", C=0.3)),
}

def top_decile_capture(y_true, p):
    k = int(np.ceil(0.10 * len(p)))
    top = np.argsort(-p)[:k]
    return y_true[top].sum() / max(y_true.sum(), 1)

oof, results = {}, {}
for name, m in models.items():
    fit_params = {"sample_weight": compute_sample_weight("balanced", y)} if name == "gbt" else {}
    p = cross_val_predict(m, X, y, cv=cv, method="predict_proba", params=fit_params)[:, 1]
    oof[name] = p
    results[name] = {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
                     "top_decile_capture": top_decile_capture(y, p)}
# naive baseline: rank by prior HIGH/CRITICAL history only (what a rules-based report would do)
naive = feat["n_anom_hc"].values + 0.001 * feat["overdue_rate"].values
results["rule_baseline"] = {"roc_auc": roc_auc_score(y, naive), "pr_auc": average_precision_score(y, naive),
                            "top_decile_capture": top_decile_capture(y, naive)}
res = pd.DataFrame(results).T.round(3)
res["lift_vs_random"] = (res["top_decile_capture"] / 0.10).round(2)
print(f"Base rate {y.mean():.3f}. Random inspection of 10% of the fleet catches 10% of events.")
display(res)

In [0]:
import mlflow
mlflow.set_experiment("/Users/ayman.el-ghazali@databricks.com/FE Bar/Aircraft Anomaly Detection")

best = max(["gbt", "logreg"], key=lambda n: results[n]["top_decile_capture"])
final = models[best]
with mlflow.start_run(run_name=f"parker_next_quarter_forecast_{best}") as run:
    if best == "gbt":
        final.fit(X, y, sample_weight=compute_sample_weight("balanced", y))
    else:
        final.fit(X, y)
    mlflow.log_params({"model": best, "cutoff": CUT, "label": "HIGH/CRITICAL anomaly or warranty-claim-risk event in next quarter",
                       "n_features": X.shape[1], "cv": "stratified 5-fold"})
    mlflow.log_metrics({f"cv_{k}": float(v) for k, v in results[best].items()})
    mlflow.log_metrics({f"rule_baseline_{k}": float(v) for k, v in results["rule_baseline"].items()})
    mlflow.sklearn.log_model(final, name="model", input_example=X.head(5))
    print("MLflow run:", run.info.run_id, "| model:", best)

out = feat[["component_id", "tail_number", "airline", "component_type", "criticality", "unit_cost_usd",
            "overdue_rate", "n_anom_hc", "cycle_life_used", "y", "n_future_events"]].copy()
out["forecast_probability"] = oof[best]          # out-of-fold score: honest for every row
out["risk_decile"] = (pd.qcut(out.forecast_probability.rank(method="first"), 10, labels=False) + 1).astype(int)  # 10 = highest
out["expected_warranty_exposure_usd"] = (out.forecast_probability * out.unit_cost_usd).round(0)
out["recommended_action"] = np.select(
    [out.risk_decile == 10, out.risk_decile >= 8], ["PROACTIVE_FIELD_SERVICE_VISIT", "FLAG_AT_NEXT_SCHEDULED_CHECK"], "ROUTINE_MONITORING")
out = out.rename(columns={"y": "actual_next_quarter_event"})
out["forecast_cutoff_date"] = pd.Timestamp(CUT).date()
out["model_run_id"] = run.info.run_id

spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUT)
spark.sql(f"COMMENT ON TABLE {OUT} IS 'Parker installed-base next-quarter failure forecast. Point-in-time features before forecast_cutoff_date; actual_next_quarter_event is the realised outcome used for back-testing. forecast_probability is out-of-fold (5-fold CV). Owner: VP Aftermarket & Customer Support.'")
display(spark.sql(f"SELECT recommended_action, count(*) components, sum(actual_next_quarter_event) events_caught, round(sum(expected_warranty_exposure_usd)) exp_exposure_usd FROM {OUT} GROUP BY 1 ORDER BY 2"))

In [0]:
# Back-test on the realised Jun-Aug 2026 quarter. Assumptions are explicit so they can be challenged.
INTERVENTION_SUCCESS = 0.60   # share of caught events a proactive visit converts to a planned repair (assumption)
f = spark.table(OUT).toPandas()
rec = spark.sql(f"""SELECT component_id, unit_cost_usd, days_undetected, flight_delay_risk
                    FROM {C}.bronze_maintenance_records m JOIN {C}.bronze_components_registry USING (component_id)
                    WHERE maintenance_date >= '{CUT}' AND ((anomaly_flag AND anomaly_severity IN ('HIGH','CRITICAL')) OR warranty_claim_risk)""").toPandas()
flagged = set(f.loc[f.risk_decile == 10, "component_id"])
rec["caught"] = rec.component_id.isin(flagged)

events_total   = len(rec)
exposure_total = rec.unit_cost_usd.sum()
caught_events  = rec.caught.sum()
caught_expo    = rec.loc[rec.caught, "unit_cost_usd"].sum()
avoided_expo   = caught_expo * INTERVENTION_SUCCESS
delay_events   = rec.flight_delay_risk.isin(["HIGH", "CRITICAL"]).sum()
delay_avoided  = (rec.caught & rec.flight_delay_risk.isin(["HIGH", "CRITICAL"])).sum() * INTERVENTION_SUCCESS
impact = pd.DataFrame([
    ["Next-quarter HIGH/CRITICAL events caught before they occur", "0 (reactive)", f"{caught_events} of {events_total} ({caught_events/events_total:.0%})"],
    ["Components Field Service must inspect to find them", "random 10% finds ~10%", "top 10% (164 components)"],
    ["Quarterly warranty exposure from these events (USD)", f"${exposure_total:,.0f}", f"${exposure_total - avoided_expo:,.0f}  (-{avoided_expo/exposure_total:.0%})"],
    ["Events with HIGH/CRITICAL flight-delay (AOG) risk", f"{delay_events}", f"{delay_events - delay_avoided:.0f}"],
    ["Mean days an anomaly goes undetected", f"{rec.days_undetected.mean():.0f} days", "0 days for flagged components (found at proactive visit)"],
], columns=["KPI", "Before (today)", f"After (top-decile proactive visits, {INTERVENTION_SUCCESS:.0%} success)"])
display(impact)
print(f"Annualised avoided warranty exposure: ${avoided_expo*4:,.0f}")

## Recorded results (run 2026-10-22, MLflow run `6b20bf8b32c84774807ed160317fa799`)

1,640 components, 37 features, next-quarter event rate 3.5% (58 components).

| Model (5-fold CV) | ROC-AUC | PR-AUC | Top-decile capture | Lift vs random |
|---|---|---|---|---|
| Logistic regression (selected) | 0.597 | 0.054 | 22.4% | 2.24x |
| Gradient-boosted trees | 0.578 | 0.052 | 15.5% | 1.55x |
| Rule: prior HIGH/CRITICAL history | 0.533 | 0.047 | 12.1% | 1.21x |

Seed robustness (logistic regression, 3 more CV seeds): AUC 0.59-0.64, capture 17-22%.

**Back-test, Jun-Aug 2026 (59 events):**

| KPI | Before | After (top-decile visits, 60% success) |
|---|---|---|
| Events caught before they occur | 0 | 14 of 59 (24%) from 164 components |
| Quarterly exposure from these events | $995,000 | $797,180 (-20%) |
| Events with HIGH/CRITICAL AOG risk | 54 | 46 |
| Annualised avoided exposure | | $791,280 |

Limitation: the simulator draws anomalies independently of service history, so overdue status carries no signal. That caps the achievable AUC. Real operator data is expected to do better.